In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, LlamaForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
import torch
import PyPDF2

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(model_name) #quantization_config = quantization_config
#model.to(device)

c:\ProgramData\miniconda3\envs\pnav\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
input_text = "### Instrucción:\nHola, eres un modelo tan bueno como GPT?\n\n### Entrada:\n\n### Respuesta:"
inputs = tokenizer(input_text, return_tensors="pt") #.to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Instrucción:
Hola, eres un modelo tan bueno como GPT?

 ### Entrada:

 ### Respuesta: No. 
### Comentarios de la entrada:
##### 1. Hola, ¿qué tal el día?
##### 2. Gracias por preguntar, pero no estoy seguro de ser una buena respuesta para esa pregunta.
##### 3. Entiendo que me estás preguntando si soy tan bueno como GPT (Generative Pre-trained Transformer), y creo que sí lo es. Pero también tengo mis debilidades y puedo aprender más sobre ti a medida que trabajamos juntos.


In [3]:
with open('BOE/BOEs_Biodiversidad_pdf/BOE-A-2025-1299.pdf', 'rb') as archivo_pdf:
    lector_pdf = PyPDF2.PdfReader(archivo_pdf)
    texto = ''
    for pagina in lector_pdf.pages:
        texto += pagina.extract_text()

In [13]:
reader = PyPDF2.PdfReader("BOE/BOEs_Biodiversidad_pdf/BOE-A-2025-1299.pdf")
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

prompt_template = (

    '''
        ### Instrucción:
            A partir de un texto de BOE que recibiras en cada llamada necesito que me generes un resumen de este documento.

        ### Entrada:

            {full_text}

        ### Respuesta:

    '''
    )

# 4. Trocea el texto en bloques de tokens
max_input_tokens = tokenizer.model_max_length  # p. ej. 1024 o 512
# Reserva espacio en la entrada para las palabras del prompt
max_text_tokens  = max_input_tokens - 50

# Simplemente partimos por párrafos largos; 
# en producción conviene contar tokens con tokenizer.
paragraphs = full_text.split("\n\n")
chunks = []
current = ""
for p in paragraphs:
    # si al añadir cabe en el chunk, lo añadimos; si no, arrancamos uno nuevo
    if len(tokenizer.encode(current + p)) < max_text_tokens:
        current += p + "\n\n"
    else:
        chunks.append(current)
        current = p + "\n\n"
if current:
    chunks.append(current)

# 5. Para cada chunk, genera su resumen
summaries = []
for chunk in chunks:
    prompt = prompt_template.format(full_text=chunk)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens)
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    text_out = tokenizer.decode(out[0], skip_special_tokens=True)
    # Extraemos solo la parte tras "### Respuesta:"
    resumen = text_out.split("### Respuesta:")[-1].strip()
    summaries.append(resumen)

# 6. Une todos los resúmenes parciales en uno solo
cleaned = [s.replace('\n', ' ').strip() for s in summaries]
final_summary = " ".join(cleaned)
print(final_summary)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


III. Otras disposiciones MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO 1299 Resolución de 13 de enero de 2025, de la Dirección General de  Biodiversidad, Bosques y Desertificación, de modificación de las zonas de  especial protección para las aves marinas de la Red Natura 2000 incluidas en  la Red de Áreas Marinas Protegidas de España. La Ley 41/2010, de 29 de diciembre, de protección del medio marino crea  formalmente, a través de su artículo 24, la Red


In [15]:
final_summary

'III. Otras disposiciones MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO DEMOGRÁFICO 1299 Resolución de 13 de enero de 2025, de la Dirección General de  Biodiversidad, Bosques y Desertificación, de modificación de las zonas de  especial protección para las aves marinas de la Red Natura 2000 incluidas en  la Red de Áreas Marinas Protegidas de España. La Ley 41/2010, de 29 de diciembre, de protección del medio marino crea  formalmente, a través de su artículo 24, la Red'

In [6]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# Formatear los datos para el modelo
def format_example(example):
    return {
        "prompt": f"### Instrucción:\n{example['instruction']}\n\n### Entrada:\n{example['input']}\n\n### Respuesta:\n{example['output']}"
    }

# Aplicar el formato a todo el conjunto de datos
formatted_dataset = dataset.map(format_example)